# TrashScan — Path A com múltiplos modelos YOLO

Este notebook foi preparado para rodar **localmente no VS Code/Jupyter**, assumindo que você vai:

1. fazer `git clone` do repositório `TrashScan`
2. abrir este notebook **de dentro do clone**
3. baixar **TACO** e **Roboflow** fora do repositório, como no fluxo anterior
4. usar o **Path A** para treinar **vários modelos YOLO** e comparar os resultados

## O que este notebook cobre

- descoberta da raiz do repositório
- configuração de caminhos locais
- merge e preprocessamento com `--path A`
- treino em grade de múltiplos modelos YOLO
- avaliação individual, em lote e com TTA/WBF
- geração de resumo global dos benchmarks

## Arquivos-base considerados

- `train_path_A.py` enviado por você, que treina uma grade de modelos e salva métricas por modelo
- `evaluate_tta.py` enviado por você, que faz avaliação unificada, suporta `--runs_dir`, TTA/WBF e resumo global

> Observação: este notebook assume a interface desses scripts como fonte da verdade.


Crie uma venv(se ja tiver pode só ativar e colocar no notebook). Ative ela. Depois intale os pacotes em /workspace/TrashScan/env/enviroment.txt.

Na parte superior direita do notebook você associa o .venv ao kernel.

## 1) Descoberta automática da raiz do repositório

A célula abaixo tenta localizar a raiz do clone do `TrashScan` mesmo se o notebook estiver dentro de `notebooks/`.


In [2]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)

REPO_ROOT = None
for p in candidates:
    if (p / "data").exists() and (p / "train").exists() and (p / "eval").exists():
        REPO_ROOT = p
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOK_DIR = cwd
PARENT_DIR = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
TRAIN_DIR = REPO_ROOT / "train" / "paths"
EVAL_DIR = REPO_ROOT / "eval"
CONFIG_DIR = REPO_ROOT / "configs"
UTILS_DIR = REPO_ROOT / "utils"

print("NOTEBOOK_DIR =", NOTEBOOK_DIR)
print("REPO_ROOT    =", REPO_ROOT)
print("PARENT_DIR   =", PARENT_DIR)


NOTEBOOK_DIR = /workspace/TrashScan/notebooks
REPO_ROOT    = /workspace/TrashScan
PARENT_DIR   = /workspace


## 2) Configuração de caminhos

Por padrão, esta versão coloca datasets e artefatos **fora do repo**, no diretório pai do clone.  
Isso evita poluir o repositório e facilita reaproveitar dados entre execuções.

Você pode ajustar essas pastas se quiser.


In [52]:
EXTERNAL_DIR = PARENT_DIR / "external_datasets"
TACO_DIR = PARENT_DIR / "TACO"
PROCESSED_DIR = PARENT_DIR / "processed_4cls"
RUNS_PATH_A_DIR = PARENT_DIR / "runs" / "path_A"
RUNS_PATH_A_DIR_EXTERNAL = PARENT_DIR.parent / "root" / "runs" / "path_A"
MLFLOW_DIR = PARENT_DIR.parent / "root" / "mlflow"
RESULTS_PATH_A_DIR = PARENT_DIR / "results_path_A"

for p in [EXTERNAL_DIR, TACO_DIR, PROCESSED_DIR, RUNS_PATH_A_DIR, RESULTS_PATH_A_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_YAML_PATH_A = PROCESSED_DIR / "dataset_path_A.yaml"

print("EXTERNAL_DIR       =", EXTERNAL_DIR)
print("TACO_DIR           =", TACO_DIR)
print("PROCESSED_DIR      =", PROCESSED_DIR)
print("RUNS_PATH_A_DIR    =", RUNS_PATH_A_DIR)
print("RESULTS_PATH_A_DIR =", RESULTS_PATH_A_DIR)
print("DATASET_YAML_PATH_A=", DATASET_YAML_PATH_A)


EXTERNAL_DIR       = /workspace/external_datasets
TACO_DIR           = /workspace/TACO
PROCESSED_DIR      = /workspace/processed_4cls
RUNS_PATH_A_DIR    = /workspace/runs/path_A
RESULTS_PATH_A_DIR = /workspace/results_path_A
DATASET_YAML_PATH_A= /workspace/processed_4cls/dataset_path_A.yaml


## 3) Verificação dos scripts usados

Aqui nós garantimos que os principais scripts do pipeline existem no clone local.


In [4]:
required_paths = [
    DATA_DIR / "merge_datasets.py",
    DATA_DIR / "preprocess.py",
    TRAIN_DIR / "train_path_A.py",
    EVAL_DIR / "evaluate.py",
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Arquivos ausentes:")
    for m in missing:
        print(" -", m)
    raise FileNotFoundError("Há scripts ausentes no clone. Veja a lista acima.")
else:
    print("Todos os scripts principais foram encontrados.")


Todos os scripts principais foram encontrados.


## 5) Utilitários de execução

As próximas células usam `subprocess` para rodar scripts do repositório de forma previsível no VS Code/Jupyter.


In [5]:
import subprocess
import shlex
import os
from pathlib import Path

def run_cmd(cmd, cwd=PARENT_DIR, env=None):
    if isinstance(cmd, str):
        print("$", cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)


## 6) GPU / CPU

O treino de vários YOLOs pode ficar pesado. Esta célula detecta CUDA e sugere um `BATCH` inicial.


In [6]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    DEVICE = "0"
    if "A100" in gpu_name:
        BATCH = 64
    elif "V100" in gpu_name:
        BATCH = 32
    else:
        BATCH = 16
else:
    gpu_name = "cpu"
    DEVICE = "cpu"
    BATCH = 8

EPOCHS = 100
IMGSZ = 640
PATIENCE = 30

print("Dispositivo:", gpu_name)
print("DEVICE     :", DEVICE)
print("BATCH      :", BATCH)
print("EPOCHS     :", EPOCHS)
print("IMGSZ      :", IMGSZ)
print("PATIENCE   :", PATIENCE)


Dispositivo: NVIDIA RTX A4500
DEVICE     : 0
BATCH      : 16
EPOCHS     : 100
IMGSZ      : 640
PATIENCE   : 30


## 9) Organização do dataset externo

No fluxo anterior, os datasets externos eram reunidos em uma pasta que depois entrava no merge.  
A célula abaixo cria o destino padrão.


Se tiver baixado os dataset acima coloque os aqui.

In [7]:
COCO_EXTERNAL_DIR = EXTERNAL_DIR / "coco_format"
COCO_EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
print("Destino esperado para datasets externos:", COCO_EXTERNAL_DIR)


Destino esperado para datasets externos: /workspace/external_datasets/coco_format


## 10) Merge dos datasets

Este passo combina TACO e os datasets externos usando o script do projeto.


In [8]:
import sys

merge_script = DATA_DIR / "merge_datasets.py"

run_cmd([
    sys.executable, str(merge_script),
    "--taco_root", str(TACO_DIR),
    "--external_root", str(EXTERNAL_DIR),
    "--output_root", str(PROCESSED_DIR),
    "--skip_preprocess",
])


$ /workspace/.venv/bin/python /workspace/TrashScan/data/merge_datasets.py --taco_root /workspace/TACO --external_root /workspace/external_datasets --output_root /workspace/processed_4cls --skip_preprocess


  TACO           : 1500 images, 4784 annotations


  TACO-dataset-1__annotations.coco:   0%|          | 0/1525 [00:00<?, ?it/s]         

  TACO-dataset-1/_annotations.coco: 189 imgs, 346 anns


  TACO-dataset-1__annotations.coco:   2%|▏         | 3/189 [00:00<00:06, 27.68it/s]    

...

  TACO-dataset-3/_annotations.coco: 461 imgs, 884 anns


  TACO-dataset-3__annotations.coco:   0%|          | 0/326 [00:00<?, ?it/s]            

  TACO-dataset-3/_annotations.coco: 2950 imgs, 6615 anns


  TACO-dataset-3/_annotations.coco: 326 imgs, 608 anns

Merged → /workspace/processed_4cls/merged_data/annotations.json
  Total images     : 9519
  Total annotations: 22362
  plastic     : 8765
  paper       : 1860
  metal       : 1793
  other       : 9944
TACO-compatible structure ready at: /workspace/processed_4cls/merged_data

Run manually:
  python preprocess.py \
      --taco_root   /workspace/processed_4cls/merged_data \
      --output_root /workspace/processed_4cls \
      --path all


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/data/merge_datasets.py', '--taco_root', '/workspace/TACO', '--external_root', '/workspace/external_datasets', '--output_root', '/workspace/processed_4cls', '--skip_preprocess'], returncode=0)

## 11) Pré-processamento para o Path A

Aqui está a troca principal em relação ao notebook anterior: agora o preprocessamento roda com `--path A`.


In [12]:
preprocess_script = DATA_DIR / "preprocess.py"

run_cmd([
    sys.executable, str(preprocess_script),
    "--taco_root", str(PROCESSED_DIR / "merged_data"),
    "--output_root", str(PROCESSED_DIR),
    "--path", "A",
])


$ /workspace/.venv/bin/python /workspace/TrashScan/data/preprocess.py --taco_root /workspace/processed_4cls/merged_data --output_root /workspace/processed_4cls --path A


INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.10). Upgrade using: pip install --upgrade albumentations


loading annotations into memory...
Done (t=0.25s)
creating index...
index created!
  Detected pre-mapped 4-class annotations — skipping TACO remapping
Class counts before merge: {'other': 3687, 'plastic': 4238, 'paper': 671, 'metal': 683}
Class counts after merge: {'other': 3687, 'plastic': 4238, 'paper': 671, 'metal': 683}
Split → train: 6495  val: 1392  test: 1392
Class weights: {'other': 0.591371, 'plastic': 0.632863, 'paper': 1.376781, 'metal': 1.398984}

[train] Processing 6495 images …


  Letterbox train: 100%|██████████| 6495/6495 [07:15<00:00, 14.91it/s]


  Running copy-paste oversampling …
  Oversampling cap: 5292  (2x median=6162)
    metal        have=1261  generate=4031  final=5292
    other        have=7057  generate=0     final=7057
    paper        have=1302  generate=3990  final=5292
    plastic      have=6162  generate=0     final=6162
  [other] already at/above cap, skipping
  [plastic] already at/above cap, skipping


...

  Letterbox test:   0%|          | 2/1392 [00:00<01:11, 19.50it/s]


[test] Processing 1392 images …


  Letterbox test: 100%|██████████| 1392/1392 [01:22<00:00, 16.88it/s]



Preprocessing complete.
Outputs written to: /workspace/processed_4cls
YOLO dataset config written: /workspace/processed_4cls/dataset_path_A.yaml


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/data/preprocess.py', '--taco_root', '/workspace/processed_4cls/merged_data', '--output_root', '/workspace/processed_4cls', '--path', 'A'], returncode=0)

## 13) Conferência do YAML do Path A

O script `train_path_A.py` espera um `--data` apontando para o YAML do dataset processado.


In [14]:
print("DATASET_YAML_PATH_A =", DATASET_YAML_PATH_A)
print("Existe?", DATASET_YAML_PATH_A.exists())

if DATASET_YAML_PATH_A.exists():
    print(DATASET_YAML_PATH_A.read_text()[:1500])
else:
    raise FileNotFoundError(
        f"Dataset YAML do Path A não encontrado em {DATASET_YAML_PATH_A}. "
        "Confirme a saída do preprocess."
    )


DATASET_YAML_PATH_A = /workspace/processed_4cls/dataset_path_A.yaml
Existe? True
path: /workspace/processed_4cls
train: train/path_A/images
val: val/path_A/images
test: test/path_A/images
nc: 4
names:
- plastic
- paper
- metal
- other



## 14) Escolha dos modelos YOLO para o grid

O `train_path_A.py` enviado por você aceita uma grade de modelos via `--models`.

Exemplos suportados no script-base:
- `yolov8n`, `yolov8s`, `yolov8m`, `yolov8l`
- `yolov9s`, `yolov9c`, `yolov9e`
- `yolov10n`, `yolov10s`, `yolov10m`
- `yolov11n`, `yolov11s`, `yolov11m`, `yolov11l`, `yolov11x`
- `yolov5su`
- `rtdetr-l`, `rtdetr-x`

Ajuste a lista abaixo conforme sua GPU e tempo disponível.


In [47]:
MODELS = [
    "yolov9s",
]

print("Modelos selecionados:", MODELS)


Modelos selecionados: ['yolov9s']


## 15) Treino do Path A para múltiplos modelos

Esta célula roda o benchmark de treino usando o script `train/paths/train_path_A.py`.

Os resultados ficam em:
- `RUNS_PATH_A_DIR/<modelo>/...`
- `mlruns/` (MLflow), se configurado pelo script


In [48]:
train_script = TRAIN_DIR / "train_path_A.py"

run_cmd([
    sys.executable, str(train_script),
    "--data", str(DATASET_YAML_PATH_A),
    "--output", str(RUNS_PATH_A_DIR),
    "--models", *MODELS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--imgsz", str(IMGSZ),
    "--device", str(DEVICE),
    "--patience", str(PATIENCE),
    "--mlflow_uri", "/root/mlflow",
])


$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_A.py --data /workspace/processed_4cls/dataset_path_A.yaml --output /workspace/runs/path_A --models yolov9s --epochs 100 --batch 16 --imgsz 640 --device 0 --patience 30 --mlflow_uri /root/mlflow


GPU  : NVIDIA RTX A4500
VRAM : 21.0 GB
Loaded class weights: {'plastic': 0.633, 'paper': 1.377, 'metal': 1.399, 'other': 0.591}

────────────────────────────────────────────────────────────
  Model  : yolov9s  (yolov9s.pt)
  Data   : /workspace/processed_4cls/dataset_path_A.yaml
  Epochs : 100   Batch : 16   imgsz : 640
────────────────────────────────────────────────────────────


100%|██████████| 14.7M/14.7M [00:00<00:00, 39.9MB/s]


New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.2 🚀 Python-3.11.10 torch-2.3.1+cu121 CUDA:0 (NVIDIA RTX A4500, 20053MiB)
engine/trainer: task=detect, mode=train, model=yolov9s.pt, data=/workspace/processed_4cls/dataset_path_A.yaml, epochs=100, time=None, patience=30, batch=16, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=8, project=/workspace/runs/path_A, name=yolov9s, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=F

train: Scanning /workspace/processed_4cls/train/path_A/labels... 14505 images, 4 backgrounds, 0 corrupt: 100%|██████████| 14505/14505 [01:15<00:00, 193.07it/s]


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:13<00:00,  3.37it/s]


                   all       1392       3175      0.663      0.528      0.584      0.391
               plastic        759       1237      0.691      0.586      0.652      0.419
                 paper        201        286      0.627      0.423      0.518      0.333
                 metal        162        244       0.61      0.577      0.559       0.37
                 other        658       1408      0.723      0.528      0.608      0.444
Speed: 0.1ms preprocess, 6.3ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /workspace/runs/path_A/yolov9s
MLflow: results logged to runs/mlflow
MLflow: disable with 'yolo settings mlflow=False'
  Latency: 49.46 ms/image


2026/04/24 16:20:56 INFO mlflow.tracking.fluent: Experiment with name 'litter_path_A' does not exist. Creating a new experiment.



  [yolov9s] mAP50=0.5843  mAP50-95=0.3915  latency=49.46ms

PATH A  —  YOLO BENCHMARK SUMMARY
          mAP50  mAP50_95  precision  recall  box_loss  cls_loss  latency_ms
model                                                                       
yolov11m 0.6620    0.4546     0.6853  0.6120    0.0000    0.0000     34.6520
yolov9s  0.5843    0.3915     0.6629  0.5285    0.0000    0.0000     49.4580


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_A.py', '--data', '/workspace/processed_4cls/dataset_path_A.yaml', '--output', '/workspace/runs/path_A', '--models', 'yolov9s', '--epochs', '100', '--batch', '16', '--imgsz', '640', '--device', '0', '--patience', '30', '--mlflow_uri', '/root/mlflow'], returncode=0)

## 16) Resumo rápido dos treinos

O próprio `train_path_A.py` permite resumir os `metrics.json` já salvos.


In [20]:
run_cmd([
    sys.executable, str(TRAIN_DIR / "train_path_A.py"),
    "--summarize",
    "--output", str(RUNS_PATH_A_DIR),
])


$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_A.py --summarize --output /workspace/runs/path_A

PATH A  —  YOLO BENCHMARK SUMMARY
          mAP50  mAP50_95  precision  recall  box_loss  cls_loss  latency_ms
model                                                                       
yolov11m 0.6620    0.4546     0.6853  0.6120    0.0000    0.0000     34.6520


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_A.py', '--summarize', '--output', '/workspace/runs/path_A'], returncode=0)

## 17) Estrutura dos pesos treinados

Verifique se os modelos produziram `weights/best.pt`.


In [49]:
best_weights = sorted(RUNS_PATH_A_DIR.glob("*/weights/best.pt"))
print(f"Pesos encontrados: {len(best_weights)}")
for p in best_weights:
    print(" -", p)


Pesos encontrados: 3
 - /workspace/runs/path_A/yolov11m/weights/best.pt
 - /workspace/runs/path_A/yolov8m/weights/best.pt
 - /workspace/runs/path_A/yolov9s/weights/best.pt


## 18) Avaliação em lote do Path A

O avaliador unificado percorre todos os `best.pt` abaixo de `--runs_dir` e produz:
- arquivos individuais por modelo
- gráficos
- resumo global do benchmark


In [50]:
evaluate_script = EVAL_DIR / "evaluate.py"

run_cmd([
    sys.executable, str(evaluate_script),
    "--path", "A",
    "--runs_dir", str(RUNS_PATH_A_DIR),
    "--data_yaml", str(DATASET_YAML_PATH_A),
    "--output", str(RESULTS_PATH_A_DIR),
    "--device", str(DEVICE),
    "--imgsz", str(IMGSZ),
])


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate.py --path A --runs_dir /workspace/runs/path_A --data_yaml /workspace/processed_4cls/dataset_path_A.yaml --output /workspace/results_path_A --device 0 --imgsz 640



────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m
  Weights   : /workspace/runs/path_A/yolov11m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:40<00:00, 13.79it/s]



  mAP@0.5    : 0.6467
  mAP@0.5:95 : 0.4181
  Precision  : 0.5682
  Recall     : 0.5362
  F1         : 0.5503
  Latency    : 10.00 ms  (100.0 FPS)
  Params     : 20.1M
  Size       : 40.5 MB
  Saved      : /workspace/results_path_A/individual/A_yolov11m_yolov11m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov8m
  Weights   : /workspace/runs/path_A/yolov8m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:15<00:00, 18.53it/s]



  mAP@0.5    : 0.6305
  mAP@0.5:95 : 0.4503
  Precision  : 0.6010
  Recall     : 0.5736
  F1         : 0.5633
  Latency    : 8.41 ms  (119.0 FPS)
  Params     : 23.2M
  Size       : 46.8 MB
  Saved      : /workspace/results_path_A/individual/A_yolov8m_yolov8m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov9s
  Weights   : /workspace/runs/path_A/yolov9s/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:16<00:00, 18.22it/s]



  mAP@0.5    : 0.5726
  mAP@0.5:95 : 0.3726
  Precision  : 0.5187
  Recall     : 0.5006
  F1         : 0.5002
  Latency    : 15.46 ms  (64.7 FPS)
  Params     : 6.3M
  Size       : 13.3 MB
  Saved      : /workspace/results_path_A/individual/A_yolov9s_yolov9s.json

GLOBAL BENCHMARK SUMMARY  —  all paths, ranked by mAP@0.5
 rank path model_key   run_id  imgsz epochs_trained stopped_early  mAP50  mAP50_95  precision  recall     f1  latency_ms      fps  model_params_M
    1    A  yolov11m yolov11m    640            100         False 0.6467    0.4181     0.5682  0.5362 0.5503     10.0020  99.9800         20.0600
    2    A   yolov8m  yolov8m    640            N/A           N/A 0.6305    0.4503     0.6010  0.5736 0.5633      8.4070 118.9500         23.2200
    3    A   yolov9s  yolov9s    640            100         False 0.5726    0.3726     0.5187  0.5006 0.5002     15.4570  64.7000          6.3200

Per-class AP@0.5:
path model_key   run_id  AP50_plastic  AP50_paper  AP50_metal  AP50_other

CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate.py', '--path', 'A', '--runs_dir', '/workspace/runs/path_A', '--data_yaml', '/workspace/processed_4cls/dataset_path_A.yaml', '--output', '/workspace/results_path_A', '--device', '0', '--imgsz', '640'], returncode=0)

## 19) Avaliação com TTA + WBF

Use esta célula para uma avaliação mais forte no inference-time.  
Ela costuma ser mais lenta, mas pode melhorar métricas.


In [44]:
# Descomente para rodar TTA + WBF
run_cmd([
    sys.executable, str(EVAL_DIR / "evaluate_tta.py"),
    "--path", "A",
    "--runs_dir", str(RUNS_PATH_A_DIR),
    "--data_yaml", str(DATASET_YAML_PATH_A),
    "--output", str(RESULTS_PATH_A_DIR),
    "--device", str(DEVICE),
    "--imgsz", str(IMGSZ),
])


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_tta.py --path A --runs_dir /workspace/runs/path_A --data_yaml /workspace/processed_4cls/dataset_path_A.yaml --output /workspace/results_path_A --device 0 --imgsz 640



────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m
  Weights   : /workspace/runs/path_A/yolov11m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:50<00:00, 12.65it/s]



  mAP@0.5    : 0.6467
  mAP@0.5:95 : 0.4181
  Precision  : 0.5682
  Recall     : 0.5362
  F1         : 0.5503
  Latency    : 10.27 ms  (97.4 FPS)
  Params     : 20.1M
  Size       : 40.5 MB
  Saved      : /workspace/results_path_A/individual/A_yolov11m_yolov11m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov8m
  Weights   : /workspace/runs/path_A/yolov8m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:20<00:00, 17.39it/s]



  mAP@0.5    : 0.6305
  mAP@0.5:95 : 0.4503
  Precision  : 0.6010
  Recall     : 0.5736
  F1         : 0.5633
  Latency    : 8.26 ms  (121.1 FPS)
  Params     : 23.2M
  Size       : 46.8 MB
  Saved      : /workspace/results_path_A/individual/A_yolov8m_yolov8m.json

GLOBAL BENCHMARK SUMMARY  —  all paths, ranked by mAP@0.5
 rank path model_key   run_id  imgsz epochs_trained stopped_early  mAP50  mAP50_95  precision  recall     f1  latency_ms      fps  model_params_M
    1    A  yolov11m yolov11m    640            100         False 0.6467    0.4181     0.5682  0.5362 0.5503     10.2670  97.4000         20.0600
    2    A   yolov8m  yolov8m    640            N/A           N/A 0.6305    0.4503     0.6010  0.5736 0.5633      8.2560 121.1200         23.2200

Per-class AP@0.5:
path model_key   run_id  AP50_plastic  AP50_paper  AP50_metal  AP50_other
   A  yolov11m yolov11m        0.7154      0.5525      0.6600      0.6588
   A   yolov8m  yolov8m        0.7938      0.3944      0.7463      0.5

CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_tta.py', '--path', 'A', '--runs_dir', '/workspace/runs/path_A', '--data_yaml', '/workspace/processed_4cls/dataset_path_A.yaml', '--output', '/workspace/results_path_A', '--device', '0', '--imgsz', '640'], returncode=0)

In [51]:
# Descomente para rodar TTA + WBF
run_cmd([
    sys.executable, str(EVAL_DIR / "evaluate_tta.py"),
    "--path", "A",
    "--runs_dir", str(RUNS_PATH_A_DIR),
    "--data_yaml", str(DATASET_YAML_PATH_A),
    "--output", str(RESULTS_PATH_A_DIR),
    "--device", str(DEVICE),
    "--imgsz", str(IMGSZ),
    "--use_tta_wbf",
    "--tta_scales", "512", "640", "768",
    "--tta_flip",
    "--tta_wbf_iou", "0.55",
    "--tta_skip_box_thr", "0.001",
])

$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_tta.py --path A --runs_dir /workspace/runs/path_A --data_yaml /workspace/processed_4cls/dataset_path_A.yaml --output /workspace/results_path_A --device 0 --imgsz 640 --use_tta_wbf --tta_scales 512 640 768 --tta_flip --tta_wbf_iou 0.55 --tta_skip_box_thr 0.001



────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m
  Weights   : /workspace/runs/path_A/yolov11m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [03:01<00:00,  7.65it/s]



  mAP@0.5    : 0.6939
  mAP@0.5:95 : 0.4467
  Precision  : 0.5811
  Recall     : 0.5301
  F1         : 0.5532
  Latency    : 9.89 ms  (101.1 FPS)
  Params     : 20.1M
  Size       : 40.5 MB
  Saved      : /workspace/results_path_A/individual/A_yolov11m_yolov11m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov8m
  Weights   : /workspace/runs/path_A/yolov8m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [02:27<00:00,  9.45it/s]



  mAP@0.5    : 0.6515
  mAP@0.5:95 : 0.4573
  Precision  : 0.6008
  Recall     : 0.5792
  F1         : 0.5684
  Latency    : 8.15 ms  (122.8 FPS)
  Params     : 23.2M
  Size       : 46.8 MB
  Saved      : /workspace/results_path_A/individual/A_yolov8m_yolov8m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov9s
  Weights   : /workspace/runs/path_A/yolov9s/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [03:35<00:00,  6.46it/s]



  mAP@0.5    : 0.6169
  mAP@0.5:95 : 0.3936
  Precision  : 0.5225
  Recall     : 0.4762
  F1         : 0.4892
  Latency    : 15.67 ms  (63.8 FPS)
  Params     : 6.3M
  Size       : 13.3 MB
  Saved      : /workspace/results_path_A/individual/A_yolov9s_yolov9s.json

GLOBAL BENCHMARK SUMMARY  —  all paths, ranked by mAP@0.5
 rank path model_key   run_id  imgsz epochs_trained stopped_early  mAP50  mAP50_95  precision  recall     f1  latency_ms      fps  model_params_M
    1    A  yolov11m yolov11m    640            100         False 0.6938    0.4467     0.5811  0.5302 0.5532      9.8900 101.1100         20.0600
    2    A   yolov8m  yolov8m    640            N/A           N/A 0.6515    0.4573     0.6008  0.5792 0.5684      8.1460 122.7700         23.2200
    3    A   yolov9s  yolov9s    640            100         False 0.6169    0.3936     0.5225  0.4762 0.4892     15.6660  63.8300          6.3200

Per-class AP@0.5:
path model_key   run_id  AP50_plastic  AP50_paper  AP50_metal  AP50_other

CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_tta.py', '--path', 'A', '--runs_dir', '/workspace/runs/path_A', '--data_yaml', '/workspace/processed_4cls/dataset_path_A.yaml', '--output', '/workspace/results_path_A', '--device', '0', '--imgsz', '640', '--use_tta_wbf', '--tta_scales', '512', '640', '768', '--tta_flip', '--tta_wbf_iou', '0.55', '--tta_skip_box_thr', '0.001'], returncode=0)